# T0.2 · Search and deduplication

Reads the Scopus, Web of Science and IEEE Xplore exports in this folder and those of the citation and author tracking of T0.4, checks the control set, removes duplicates and writes `corpus.csv` and `counts.csv`.

- **Input:** `Scopus/*.csv`, `WOS/*.xls`, `IEEE/X1/*.csv`, `IEEE/X2/*.csv`, the `query_*` folders of T0.4 (if they hold exports) and `../T0.1-protocol/control_set.csv`.
- **To re-run:** replace the exports and run all cells. Identifiers `R0001`… already assigned are kept.
- **Environment:** the repository `.venv` (see `CONTRIBUTING.md`).

In [1]:
import re, unicodedata, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 90)
CONTROL = Path("../T0.1-protocol/control_set.csv")
CORPUS_THRESHOLD = 2000   # protocol: above this the broad terms are restricted (only the database search counts)
T04 = Path("../T0.4-citation-and-author-tracking")
T04_SOURCES = {"backward_citations": "query_backward_citations", "forward_citations": "query_forward_citations", "author": "query_authors"}
DATABASES = ["scopus", "wos", "ieee"]

## 1. Loading the exports

In [2]:
def read_csvs(folder):
    files = sorted(Path(folder).glob("*.csv"))
    return pd.concat([pd.read_csv(f, dtype=str, encoding="utf-8-sig") for f in files]).fillna("")

def read_xls(folder):
    files = sorted(Path(folder).glob("*.xls"))
    return pd.concat([pd.read_excel(f, dtype=str) for f in files]).fillna("")

sc = read_csvs("Scopus")
wos = read_xls("WOS")
x1 = read_csvs("IEEE/X1")
x2 = read_csvs("IEEE/X2")

def join(*columns):
    return pd.concat(columns, axis=1).apply(lambda r: "; ".join(v for v in r if v), axis=1)

def from_ieee(df, query):
    return pd.DataFrame({
        "database": "ieee", "query": query, "db_id": df["PDF Link"],
        "title": df["Document Title"], "authors": df["Authors"], "year": df["Publication Year"],
        "source": df["Publication Title"], "type": df["Document Identifier"], "language": "",
        "doi": df["DOI"], "abstract": df["Abstract"],
        "keywords": join(df["Author Keywords"], df["IEEE Terms"]),
    })

def from_scopus(df, database, query):
    return pd.DataFrame({
        "database": database, "query": query, "db_id": df["EID"],
        "title": df["Title"], "authors": df["Authors"], "year": df["Year"],
        "source": df["Source title"], "type": df["Document Type"],
        "language": df["Language of Original Document"], "doi": df["DOI"], "abstract": df["Abstract"],
        "keywords": join(df["Author Keywords"], df["Index Keywords"]),
    })

# Scopus exports of the citation and author tracking (T0.4), if any
tracking = [from_scopus(read_csvs(T04 / folder), database, "tracking")
            for database, folder in T04_SOURCES.items() if list((T04 / folder).glob("*.csv"))]

raw = pd.concat([
    from_scopus(sc, "scopus", "main"),
    pd.DataFrame({
        "database": "wos", "query": "main", "db_id": wos["UT (Unique WOS ID)"],
        "title": wos["Article Title"], "authors": wos["Authors"],
        "year": wos["Publication Year"].str.split(".").str[0],
        "source": wos["Source Title"], "type": wos["Document Type"], "language": wos["Language"],
        "doi": wos["DOI"], "abstract": wos["Abstract"],
        "keywords": join(wos["Author Keywords"], wos["Keywords Plus"]),
    }),
    from_ieee(x1, "X1"), from_ieee(x2, "X2"),
    *tracking,
], ignore_index=True)

raw["doi"] = raw["doi"].str.strip().str.lower().str.replace(r"^https?://(dx\.)?doi\.org/", "", regex=True)
raw["abstract"] = raw["abstract"].replace("[No abstract available]", "")

exported = raw.groupby(["database", "query"], sort=False).size().rename("records").reset_index()
exported

,database,query,records
0,scopus,main,1380
1,wos,main,801
2,ieee,X1,545
3,ieee,X2,488
4,forward_citations,tracking,692
5,author,tracking,422


## 2. Front matter and proceedings

IEEE Xplore returns front covers, tables of contents, whole proceedings volumes and standards. Fixed rule: every record without authors is dropped. One more applies to the tracking records: only articles, conference papers, reviews and book chapters enter, as in the Scopus string.

In [3]:
front_matter = raw["authors"].str.strip().isin(["", "[No author name available]"])
other_type = (raw["query"] == "tracking") & ~raw["type"].isin(["Article", "Conference paper", "Review", "Book chapter"]) & ~front_matter
print("Tracking, dropped by document type:", int(other_type.sum()))
print(front_matter.groupby(raw["database"]).sum().rename("dropped for having no authors"))
raw[front_matter][["database", "title"]].drop_duplicates().sort_values("database", ascending=False).head(12)

Tracking, dropped by document type: 7
database
author                0
forward_citations     0
ieee                 71
scopus                3
wos                   0
Name: dropped for having no authors, dtype: int64


,database,title
655,scopus,Enhanced performance through DSP
1285,scopus,"PROCEEDINGS - IEEE INTERNATIONAL CONFERENCE ON CIRCUITS AND COMPUTERS, 1980."
708,scopus,2009 Product of the year
3207,ieee,15th Asian Test Symposium - Title
3206,ieee,IEEE 15th International Symposium on Design and Diagnostics of Electronic Circuits & S...
3205,ieee,Proceedings 21st IEEE VLSI Test Symposium
3196,ieee,[Cover art]
3195,ieee,Proceedings International Test Conference 2008
3193,ieee,25th IEEE VLSI Test Symmposium - Title page
3079,ieee,Proceedings 20th IEEE VLSI Test Symposium (VTS 2002)


In [4]:
rec = raw[~front_matter & ~other_type].reset_index(drop=True)

## 3. Deduplication

1. Same DOI: same record.
2. Same normalised title, same or adjacent year and no conflicting DOIs: same record.
3. Same title with different DOIs: **not** merged. They are flagged with a `possible_version` group (conference and journal version, or a mistyped DOI) and decided at screening, reason E6.

In [5]:
def title_key(t):
    t = re.sub(r"\[[^\]]*\]\s*$", "", t)                      # original title in square brackets (Scopus)
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]", "", t)

rec["title_key"] = rec["title"].map(title_key)
rec["year_n"] = pd.to_numeric(rec["year"], errors="coerce")

parent = list(range(len(rec)))
def root(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def merge(i, j):
    parent[root(i)] = root(j)

# Step 1: by DOI
for _, idx in rec[rec["doi"] != ""].groupby("doi").groups.items():
    for i in idx[1:]:
        merge(i, idx[0])

# Step 2: by title, if the DOIs do not conflict and the year is compatible
def dois(group):
    return {rec.at[i, "doi"] for i in group if rec.at[i, "doi"]}
def years(group):
    return {rec.at[i, "year_n"] for i in group if pd.notna(rec.at[i, "year_n"])}

for key, idx in rec[rec["title_key"].str.len() >= 25].groupby("title_key").groups.items():
    idx = list(idx)
    for a in idx:
        for b in idx:
            if root(a) == root(b):
                continue
            ga = [i for i in idx if root(i) == root(a)]
            gb = [i for i in idx if root(i) == root(b)]
            da, db = dois(ga), dois(gb)
            ya, yb = years(ga), years(gb)
            compatible_doi = not (da and db)
            compatible_year = not (ya and yb) or min(abs(p - q) for p in ya for q in yb) <= 1
            if compatible_doi and compatible_year:
                merge(a, b)

rec["group"] = [root(i) for i in range(len(rec))]
print("records with authors:", len(rec), "| unique:", rec["group"].nunique())

records with authors: 4247 | unique: 2632


In [6]:
# Scopus first: it carries the references for T0.4. Tracking goes last so that existing records are not altered
PRIORITY = {b: n for n, b in enumerate(DATABASES + list(T04_SOURCES))}
IN_SCOPUS = ["scopus"] + list(T04_SOURCES)

def first_non_empty(series):
    return next((v for v in series if v), "")

rows = []
for _, g in rec.sort_values("database", key=lambda s: s.map(PRIORITY)).groupby("group", sort=False):
    rows.append({
        "title": g["title"].iloc[0], "authors": first_non_empty(g["authors"]),
        "year": first_non_empty(g["year"]), "source": first_non_empty(g["source"]),
        "type": first_non_empty(g["type"]), "language": first_non_empty(g["language"]),
        "doi": first_non_empty(g["doi"]),
        "abstract": max(g["abstract"], key=len),
        "keywords": "; ".join(v for v in g["keywords"] if v),
        "databases": "+".join(sorted(set(g["database"]), key=PRIORITY.get)),
        "id_scopus": first_non_empty(g.loc[g["database"].isin(IN_SCOPUS), "db_id"]),
        "id_wos": first_non_empty(g.loc[g["database"] == "wos", "db_id"]),
        "id_ieee": first_non_empty(g.loc[g["database"] == "ieee", "db_id"]),
        "title_key": g["title_key"].iloc[0],
    })
corpus = pd.DataFrame(rows)

# Groups of possible redundant versions
rep = corpus[corpus["title_key"].str.len() >= 25].groupby("title_key")["title"].transform("size") > 1
codes = {c: f"V{n:03d}" for n, c in enumerate(sorted(corpus.loc[rep[rep].index, "title_key"].unique()), 1)}
corpus["possible_version"] = corpus["title_key"].where(corpus.index.isin(rep[rep].index), "").map(lambda c: codes.get(c, ""))
print("records in possible-version groups:", (corpus["possible_version"] != "").sum(), "in", len(codes), "groups")
corpus[corpus["possible_version"] != ""].sort_values("possible_version")[["possible_version", "year", "doi", "source", "title"]].head(12)

records in possible-version groups: 99 in 48 groups


,possible_version,year,doi,source,title
929,V001,2015,10.1109/tcad.2015.2440315,IEEE Transactions on Computer-Aided Design of Integrated Circuits and Systems,Accurate QBF-Based Test Pattern Generation in Presence of Unknown Values
1285,V001,2013,10.7873/date.2013.098,"Proceedings -Design, Automation and Test in Europe, DATE",Accurate QBF-based test pattern generation in presence of unknown values
1303,V002,2006,10.1109/imtc.2006.235517,Conference Record - IEEE Instrumentation and Measurement Technology Conference,A fault diagnosis method of analog electronic circuits for mixed-signal systems contro...
1373,V002,2006,10.1109/imtc.2006.328453,"2006 IEEE INSTRUMENTATION AND MEASUREMENT TECHNOLOGY CONFERENCE PROCEEDINGS, VOLS 1-5",A fault diagnosis method of analog electronic circuits for mixed-signal systems contro...
1483,V003,2003,10.1109/dftvs.2003.1250125,"18TH IEEE INTERNATIONAL SYMPOSIUM ON DEFECT AND FAULT TOLERANCE IN VLSI SYSTEMS, PROCE...",A monolithic spectral BIST technique for control or test of analog or mixed-signal cir...
35,V003,2003,10.1109/tsm.2005.1250125,Proceedings - IEEE International Symposium on Defect and Fault Tolerance in VLSI Systems,A monolithic spectral BIST technique for control or test of analog or mixed-signal cir...
99,V004,2000,10.1145/368434.368784,"Proceedings of the Asia and South Pacific Design Automation Conference, ASP-DAC",Analog-testability analysis by determinant-decision-diagrams based symbolic analysis
1749,V004,2000,10.1109/aspdac.2000.835160,Proceedings 2000. Design Automation Conference. (IEEE Cat. No.00CH37106),Analog testability analysis by determinant-decision-diagrams based symbolic analysis
1461,V005,2003,10.1109/dftvs.2003.1250123,"18TH IEEE INTERNATIONAL SYMPOSIUM ON DEFECT AND FAULT TOLERANCE IN VLSI SYSTEMS, PROCE...",An approach for selection of test points for analog fault diagnosis
231,V005,2003,10.1109/tsm.2005.1250123,Proceedings - IEEE International Symposium on Defect and Fault Tolerance in VLSI Systems,An approach for selection of test points for analog fault diagnosis


## 4. Control set

In [7]:
control = pd.read_csv(CONTROL, dtype=str).fillna("")
control["doi"] = control["doi"].str.lower()

def retrieved(df, k):
    if k["doi"]:
        return (df["doi"] == k["doi"]).any()
    return df["title"].str.lower().str.contains(k["title_contains"], regex=False).any()

table = control[["id", "reference"]].copy()
for database in DATABASES:
    table[database] = [retrieved(raw[raw["database"] == database], k) for _, k in control.iterrows()]
table["any"] = table[DATABASES].any(axis=1)

print(table[DATABASES + ["any"]].sum().rename("retrieved out of %d" % len(table)))
table[~table[DATABASES].all(axis=1)].replace({True: "yes", False: "—"})

scopus    22
wos       19
ieee      17
any       22
Name: retrieved out of 24, dtype: int64


,id,reference,scopus,wos,ieee,any
3,K04,"Tadeusiewicz and Hałgas, JETTA 2017",—,—,—,—
4,K05,"Brosa and Figueras, ETW 1999",yes,—,yes,yes
5,K06,"Zilch, Lubaszewski and Balen, AICSP 2022",yes,yes,—,yes
7,K08,"Spinks et al., JETTA 2004",yes,yes,—,yes
8,K09,"Grzechca, ECCTD 2011",yes,—,yes,yes
9,K10,"Yang et al., EAAI 2026",yes,yes,—,yes
11,K12,"Ankush et al. (CODA), ETS 2026",yes,—,yes,yes
13,K14,"Lin and Elcherif, IJCTA 1985",yes,yes,—,yes
14,K15,"Stenbakken and Souders, IEEE TIM 1987",—,—,—,—
23,K24,"Ma et al., Math. Probl. Eng. 2018",yes,yes,—,yes


## 5. Identifiers and output files

Identifiers are assigned by year and title. If `corpus.csv` already exists, every record keeps its own and new ones get the following numbers. A record is recognised by its DOI or, if it had none before, by title and year.

In [8]:
corpus["key_t"] = "t:" + corpus["title_key"] + ":" + corpus["year"]
corpus["key"] = corpus["doi"].where(corpus["doi"] != "", corpus["key_t"])
corpus = corpus.sort_values(["year", "title_key", "doi"]).reset_index(drop=True)

previous, earlier = {}, set()
if Path("corpus.csv").exists():
    old = pd.read_csv("corpus.csv", dtype=str).fillna("")
    previous = dict(zip(old["key"], old["id"]))
    earlier = set(old["id"])
next_n = max([int(i[1:]) for i in previous.values()], default=0) + 1
ids = []
for k, kt in zip(corpus["key"], corpus["key_t"]):
    rid = previous.get(k) or previous.get(kt)
    if rid is None:
        rid = f"R{next_n:04d}"
        next_n += 1
    ids.append(rid)
corpus["id"] = ids
assert corpus["id"].is_unique
lost = earlier - set(ids)
assert not lost, f"identifiers that disappear: {sorted(lost)[:10]}"
print("New records with respect to the previous corpus:", len(set(ids) - earlier))

columns = ["id", "databases", "year", "title", "authors", "source", "type", "language", "doi",
           "possible_version", "abstract", "keywords", "id_scopus", "id_wos", "id_ieee", "key"]
corpus.sort_values("id")[columns].to_csv("corpus.csv", index=False, encoding="utf-8")

from_databases = corpus["databases"].str.contains("|".join(DATABASES))
counts = pd.DataFrame([
    ("Scopus", (raw["database"] == "scopus").sum()),
    ("Web of Science", (raw["database"] == "wos").sum()),
    ("IEEE Xplore X1", ((raw["database"] == "ieee") & (raw["query"] == "X1")).sum()),
    ("IEEE Xplore X2", ((raw["database"] == "ieee") & (raw["query"] == "X2")).sum()),
    *[(f"Tracking: {database.replace('_', ' ')}", (raw["database"] == database).sum()) for database in T04_SOURCES],
    ("Total identified", len(raw)),
    ("Removed: no authors (front matter, proceedings, standards)", int(front_matter.sum())),
    ("Removed: document type (tracking)", int(other_type.sum())),
    ("Removed: duplicates", len(rec) - len(corpus)),
    ("Corpus for screening", len(corpus)),
    ("  of which, from the database searches", int(from_databases.sum())),
    ("  of which, from tracking only", int((~from_databases).sum())),
], columns=["item", "records"])
counts.to_csv("counts.csv", index=False, encoding="utf-8")
if from_databases.sum() > CORPUS_THRESHOLD:
    print(f"WARNING: the corpus exceeds the protocol threshold of {CORPUS_THRESHOLD}")
counts

New records with respect to the previous corpus: 0


,item,records
0,Scopus,1380
1,Web of Science,801
2,IEEE Xplore X1,545
3,IEEE Xplore X2,488
4,Tracking: backward citations,0
5,Tracking: forward citations,692
6,Tracking: author,422
7,Total identified,4328
8,"Removed: no authors (front matter, proceedings, standards)",74
9,Removed: document type (tracking),7


## 6. Corpus profile

In [9]:
print("By combination of databases")
print(corpus["databases"].value_counts().to_string(), "\n")
decade = (pd.to_numeric(corpus["year"], errors="coerce") // 10 * 10).astype("Int64")
print("By decade")
print(decade.value_counts().sort_index().to_string(), "\n")
print("No abstract:", (corpus["abstract"].str.len() < 50).sum())
print("No DOI:", (corpus["doi"] == "").sum())
print("Language other than English (according to Scopus or WoS):",
      (~corpus["language"].str.lower().isin(["", "english", "undefined"])).sum())

By combination of databases
databases
scopus                                      476
author                                      369
ieee                                        368
forward_citations                           366
scopus+wos                                  264
scopus+wos+ieee                             192
scopus+ieee                                 155
scopus+wos+forward_citations                102
scopus+wos+ieee+forward_citations            86
wos                                          80
scopus+forward_citations                     42
wos+forward_citations                        32
scopus+ieee+forward_citations                19
wos+ieee                                     19
forward_citations+author                     15
ieee+forward_citations                        9
scopus+wos+ieee+author                        7
scopus+ieee+author                            6
wos+ieee+forward_citations                    5
ieee+author                                   4
sc

## Conclusions and notes

- **Corpus for screening: 2632 records.** 1882 come from the database searches (below the threshold of 2000) and 750 only from the citation and author tracking of T0.4.
- **Cleaning:** of 4328 records identified, 74 without authors, 7 tracking records by document type and 1615 duplicates are removed.
- **Control set: 23 of 24 in the corpus.** The database searches retrieve 22 (Scopus 22, WoS 19, IEEE Xplore 17). K04 (Tadeusiewicz and Hałgas) enters through tracking; K15 (Stenbakken and Souders) is still waiting for the backward citations.
- **Possible redundant versions: 99 records in 48 groups.** They are not merged; they are decided at full text (reason E6).
- **Language: 91 records** are listed as non-English. They are screened by the abstract.
- **No abstract: 14 records.** They are decided by title or at full text.
- **Identifiers:** the 1882 earlier records keep theirs; the new ones run from R1883 to R2632.

### Notes